<a href="https://colab.research.google.com/github/deepakawl/supplychain-analytics-teaching/blob/main/Biopharma_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Biopharma

### Install and import packages

In [2]:
%reset -f
# Install and import packages
!pip install forex_python
!pip install gurobipy
# !pip install tabulate

import pandas as pd
import numpy as np
from gurobipy import Model, GRB, quicksum
from tabulate import tabulate
import datetime as dt
from forex_python.converter import get_rate
_empty_series = pd.Series(dtype=float)

### Raw Data

In [4]:
# selected_yr = 2023
base_yr = 2013

demand = pd.DataFrame({
    'from': ['LatinAmerica', 'Europe', 'AsiaWoJapan', 'Japan', 'Mexico', 'U.S.'],
    'd_h': [  7, 15,  5,  7,  3, 18],
    'd_r': [  7, 12,  3,  8,  3, 17],
})
demand.set_index('from', inplace=True)

caps = pd.DataFrame({
    'plant': ['Brazil', 'Germany', 'India', 'Japan', 'Mexico', 'U.S.'],
    'cap': [18, 45, 18, 10, 30, 22],
})
caps.set_index('plant', inplace=True)

pcosts = pd.DataFrame({
    'plant': ['Brazil', 'Germany', 'India', 'Japan', 'Mexico', 'U.S.'],
    'fc_p': [20, 45, 14, 13, 30, 23],
    'fc_h': [ 5, 13,  3,  4,  6,  5],
    'fc_r': [ 5, 13,  3,  4,  6,  5],
    'rm_h': [3.6, 3.9, 3.6, 3.9, 3.6, 3.6],
    'pc_h': [5.1, 6.0, 4.5, 6.0, 5.0, 5.0],
    'rm_r': [4.6, 5.0, 4.5, 5.1, 4.6, 4.5],
    'pc_r': [6.6, 7.0, 6.0, 7.0, 6.5, 6.5],
})
pcosts.set_index('plant', inplace=True)

tcosts = pd.DataFrame({
    'from': ['Brazil', 'Germany', 'India', 'Japan', 'Mexico', 'U.S.'],
    'LatinAmerica': [ 0.20, 0.45, 0.50, 0.50, 0.40, 0.45],
    'Europe':       [ 0.45, 0.20, 0.35, 0.40, 0.30, 0.30],
    'AsiaWoJapan':  [ 0.50, 0.35, 0.20, 0.30, 0.50, 0.45],
    'Japan':        [ 0.50, 0.40, 0.30, 0.10, 0.45, 0.45],
    'Mexico':       [ 0.40, 0.30, 0.50, 0.45, 0.20, 0.25],
    'U.S.':           [ 0.45, 0.30, 0.45, 0.45, 0.25, 0.20],
})
tcosts.set_index('from', inplace=True)

duties = pd.DataFrame({
    'from': ['LatinAmerica', 'Europe', 'AsiaWoJapan', 'Japan', 'Mexico', 'U.S.'],
    'duty': [ 0.30, 0.03, 0.27, 0.06, 0.35, 0.04],
})
duties.set_index('from', inplace=True)

# Your provided exchange_rate_data
exrate0 = {
    '2013': [2.15, 0.75, 58.44, 97.58, 12.75, 1],
    '2017': [3.28, 0.96, 68.13, 117.5, 20.74, 1],
    '2018': [3.26, 0.83, 63.46, 112.24, 19.5, 1],
    '2019': [3.79, 0.88, 69.96, 107.44, 19.61, 1],
    '2020': [4.02, 0.9, 71.42, 108.54, 18.83, 1],
    '2021': [5.19, 0.82, 73.09, 103.24, 19.87, 1],
    '2022': [5.57, 0.88, 74.4, 115.14, 20.48, 1],
    '2023': [5.36, 0.94, 82.71, 130.79, 19.46, 1],
    '2024': [4.92, 0.91, 83.26, 142.16, 17.02, 1],
    '2025': [6.15, 0.97, 85.8, 157.36, 20.59, 1],
    '2026': [5.19, 0.87, 92.39, 158.91, 17.66, 1],
}
exrate0 = pd.DataFrame(exrate0 , index=['BRL', 'EUR', 'INR', 'JPY', 'MXN', 'USD'])

# Extract Currency Exchange Rates

In [5]:
################ Function to extract currency exchange rates
def extract_exrate(base_yr, selected_yr):
    currency_list = ["BRL", "EUR", "INR", "JPY", "MXN", "USD"]
    years = [base_yr, selected_yr]

    # # Create DataFrame to store exchange rates
    exrate = pd.DataFrame(index=currency_list, columns=years)

    # Populate DataFrame with exchange rates
    for currency in currency_list:
      for year in years:
        last_day = dt.date(year, 12, 31)
        exrate.at[currency, year] = round(get_rate("USD", currency, last_day), 2)

    del currency, currency_list, year, years, last_day
    exrate.index = exrate.index.astype(str)
    exrate.columns = exrate.columns.astype(str)

    return exrate

# Minimize cost using Gurobi Binary and Integer optimizer

## Functions to calculate cost, unmet demand, and excess capacity

In [6]:
# identify number of supply and demand location for iterations
n_ctry = range(demand.shape[0])
n_lines = range(demand.shape[1]+1)

# Objective function to calculate cost
def calc_total_cost(dec_plant, dec_h, dec_r, base_yr=2019, selected_yr=2023, tariff=0):
    x_plant = np.array(list(dec_plant.values())).reshape(len(n_ctry), len(n_lines))
    x_h = np.array(list(dec_h.values())).reshape(len(n_ctry), len(n_ctry))
    x_r = np.array(list(dec_r.values())).reshape(len(n_ctry), len(n_ctry))

    if all(col == selected_yr for col in exrate0.columns):
        exrate = exrate0.copy()
    else:
        exrate = extract_exrate(base_yr, selected_yr)

    # Adjust the cost using exchange rate of give year
    if (base_yr == selected_yr):
        reindx = pd.Series(1, index=exrate.index)
    else:
        reindx = exrate.loc[:, f'{base_yr}'] / exrate.loc[:, f'{selected_yr}']

    pcosts_rev = pcosts.values * reindx.values.reshape(-1,1)
    pcosts_rev = pd.DataFrame(pcosts_rev, columns=pcosts.columns[0:], index=pcosts.index)

    duties_mat = np.zeros(len(duties)) + (1 + duties['duty'].values)[:, np.newaxis]
    np.fill_diagonal(duties_mat, 1)
    duties_mat = pd.DataFrame(duties_mat.T, index=pcosts_rev.index, columns=duties.index)
    duties_mat.loc['Germany', 'U.S.'] += tariff
    duties_mat.loc['U.S.', 'Europe']  += tariff

    vcosts_h = tcosts.add(pcosts_rev['rm_h'], axis=0).add(pcosts_rev['pc_h'], axis=0) * duties_mat
    vcosts_r = tcosts.add(pcosts_rev['rm_r'], axis=0).add(pcosts_rev['pc_r'], axis=0) * duties_mat

    fc = pcosts_rev[['fc_p','fc_h','fc_r']].values
    vh = (vcosts_h * x_h).values
    vr = (vcosts_r * x_r).values
    total_cost = sum(0.2 * fc[i,j] for i in n_ctry for j in n_lines) + sum(0.8 * fc[i,j] * x_plant[i,j] for i in n_ctry for j in n_lines) + sum(vh[i,j] for i in n_ctry for j in n_ctry) + sum(vr[i,j] for i in n_ctry for j in n_ctry)

    return total_cost


# Calculate excess capacity given decision variables
def calc_excess_cap(dec_plant, dec_h, dec_r):
    x_plant = np.array(list(dec_plant.values())).reshape(len(n_ctry), len(n_lines))
    x_h = np.array(list(dec_h.values())).reshape(len(n_ctry), len(n_ctry))
    x_r = np.array(list(dec_r.values())).reshape(len(n_ctry), len(n_ctry))

    excess_cap = (x_plant * caps.values).copy()
    excess_cap[:, 0] -= (np.sum(x_h, axis=1) + np.sum(x_r, axis=1))
    excess_cap[:, 1] -= np.sum(x_h, axis=1)
    excess_cap[:, 2] -= np.sum(x_r, axis=1)
    return excess_cap

# Calculate unmet demand given decision variables
def calc_unmet_demand(dec_h, dec_r):
    x_h = np.array(list(dec_h.values())).reshape(len(n_ctry), len(n_ctry))
    x_r = np.array(list(dec_r.values())).reshape(len(n_ctry), len(n_ctry))

    x_h_sum = np.sum(x_h, axis=0)
    x_r_sum = np.sum(x_r, axis=0)
    unmet_demand = (demand.values).copy()
    unmet_demand = np.column_stack((x_h_sum - unmet_demand[:, 0], x_r_sum - unmet_demand[:, 1]))

    return unmet_demand


## Gurobi optimizer

In [8]:
# Prompt the user to enter the year
while True:
    try:
        selected_yr = int(input("Enter year (as yyyy, 2026): "))
        if 1995 <= selected_yr <= 2026:
          if selected_yr > 2026:
            print("Cannot fetch forecasted value.")
            break
          if selected_yr < 2017:
            print("Extracting exchange rate from the web. Please hold.")
          break  # Break the loop if the input is valid
        else:
            print("Invalid input. Please enter a year between 1995 and 2026.")
    except ValueError:
        print("Invalid input. Please enter a valid year in number (yyyy).")

while True:
    try:
        tariff = float(input("Enter tariff (in percent, e.g. 10 for 10%): "))
        if 0 <= tariff <= 200:
          tariff = tariff/100
          break  # Break the loop if the input is valid
        else:
            print("Invalid input. Please enter a valid number between 0 and 200.")
    except ValueError:
        print("Invalid input. Please enter a valid number.")


# Create a Gurobi model
model = Model("MinimizeCost")

# Assign initial value of decision variables
dec_plant = {(i, j): 1 for i in n_ctry for j in n_lines}
dec_h     = {(i, j): 1 for i in n_ctry for j in n_ctry}
dec_r     = {(i, j): 1 for i in n_ctry for j in n_ctry}

# Define decision variables
dec_plant = {(i, j): model.addVar(vtype=GRB.BINARY, name=f"Dec_plant_{i}_{j}")    for i in n_ctry for j in n_lines}
dec_h     = {(i, j): model.addVar(vtype=GRB.INTEGER, lb=0, name=f"Dec_h_{i}_{j}") for i in n_ctry for j in n_ctry}
dec_r     = {(i, j): model.addVar(vtype=GRB.INTEGER, lb=0, name=f"Dec_r_{i}_{j}") for i in n_ctry for j in n_ctry}

# Excess Capacity constraints
excess_cap = calc_excess_cap(dec_plant, dec_h, dec_r)
for i in n_ctry:
    for j in n_lines:
        model.addConstr(excess_cap[i, j] >= 0, name=f"Excess_Cap_Constraints_{i}_{j}")


# Unmet demand constraints
unnmet_demand = calc_unmet_demand(dec_h, dec_r)
for i in n_ctry:
    for j in range(2):
        model.addConstr(unnmet_demand[i,j] == 0, name=f"Unmet_Demand_Constraints_{i}_{j}")


# Update the model
model.update()

# Set objective function - Total cost = Fixed cost + Variable costs of Highcal and Relax lines
model.setObjective(calc_total_cost(dec_plant, dec_h, dec_r, base_yr, selected_yr, tariff), GRB.MINIMIZE)

# Suppress optimization output
model.Params.OutputFlag = 0

# Optimize the model
model.optimize()

# Extract results to print as table
op_plant = pd.DataFrame([[dec_plant[i, j].x for j in n_lines] for i in n_ctry], columns = ['Plant','H','R'], index=caps.index)
op_h     = pd.DataFrame([[dec_h[i, j].x for j in n_ctry] for i in n_ctry], columns = tcosts.columns, index=tcosts.index)
op_r     = pd.DataFrame([[dec_r[i, j].x for j in n_ctry] for i in n_ctry], columns = tcosts.columns, index=tcosts.index)


print("\nHighCal Flow\n")
print(tabulate(op_h, headers='keys', tablefmt='pretty'))
print("\nRelax Flow\n")
print(tabulate(op_r, headers='keys', tablefmt='pretty'))
print("\nStrategy\n")
print(tabulate(op_plant, headers='keys', tablefmt='pretty'))
print(f"\nMinimum Cost: $ {round(model.objVal,2)} in year {selected_yr} at Tariff {(tariff*100)}")

Enter year (as yyyy, 2026): 2025
Enter tariff (in percent, e.g. 10 for 10%): 10
Restricted license - for non-production use only - expires 2027-11-29

HighCal Flow

+---------+--------------+--------+-------------+-------+--------+------+
|  from   | LatinAmerica | Europe | AsiaWoJapan | Japan | Mexico | U.S. |
+---------+--------------+--------+-------------+-------+--------+------+
| Brazil  |     7.0      |  -0.0  |    -0.0     | -0.0  |  -0.0  | -0.0 |
| Germany |     -0.0     |  15.0  |    -0.0     |  2.0  |  -0.0  | -0.0 |
|  India  |     -0.0     |  -0.0  |     5.0     |  5.0  |  -0.0  | -0.0 |
|  Japan  |     -0.0     |  -0.0  |    -0.0     | -0.0  |  -0.0  | -0.0 |
| Mexico  |     -0.0     |  0.0   |    -0.0     | -0.0  |  3.0   | 18.0 |
|  U.S.   |     -0.0     |  -0.0  |    -0.0     | -0.0  |  -0.0  | 0.0  |
+---------+--------------+--------+-------------+-------+--------+------+

Relax Flow

+---------+--------------+--------+-------------+-------+--------+------+
|  from 

### Run for multiple years and tariff and Store result into csv file

In [9]:
tariffs = [0, 5, 10]
year_range = range(2017, 2026)

op_df = op_plant.iloc[:, 1:]
op_df = op_df.stack().reset_index().rename(columns={'level_0': 'index_name', 'level_1': 'suffix', 0: 'value'})
op_df['column_name'] = op_df['plant'] + '-' + op_df['suffix']
op_df = op_df[['column_name', 'value']].set_index('column_name').T.reset_index(drop=True)
op_df['Year'] = selected_yr
op_df['Cost'] = round(model.objVal,2)
op_df['Tariff'] = tariff


result_df = op_df.copy().iloc[0:0]
for tariff in tariffs:
    for selected_yr in year_range:
        print(f"Running for {selected_yr} with tariff {tariff}")

        # Set objective function - Total cost = Fixed cost + Variable costs of Highcal and Relax lines
        model.setObjective(calc_total_cost(dec_plant, dec_h, dec_r, base_yr, selected_yr, tariff), GRB.MINIMIZE)

        # Optimize the model
        model.optimize()

        op_plant = pd.DataFrame([[dec_plant[i, j].x for j in n_lines] for i in n_ctry], columns = ['Plant','H','R'], index=caps.index)
        op_df = op_plant.iloc[:, 1:]
        op_df = op_df.stack().reset_index().rename(columns={'level_0': 'index_name', 'level_1': 'suffix', 0: 'value'})
        op_df['column_name'] = op_df['plant'] + '-' + op_df['suffix']
        op_df = op_df[['column_name', 'value']].set_index('column_name').T.reset_index(drop=True)
        op_df['Year'] = selected_yr
        op_df['Cost'] = round(model.objVal,2)
        op_df['Tariff'] = tariff

        result_df = pd.concat([result_df, op_df], ignore_index=True)
        print(round(model.objVal,2), "\n")

result_df.to_csv(f'Result.csv', index=False)
print("Results are saved in Result.csv file")

Running for 2017 with tariff 0
1039.42 

Running for 2018 with tariff 0
983.76 

Running for 2019 with tariff 0
975.9 

Running for 2020 with tariff 0
967.68 

Running for 2021 with tariff 0
918.33 

Running for 2022 with tariff 0
892.64 

Running for 2023 with tariff 0
946.71 

Running for 2024 with tariff 0
842.44 

Running for 2025 with tariff 0
917.3 

Running for 2017 with tariff 5
1041.83 

Running for 2018 with tariff 5
986.23 

Running for 2019 with tariff 5
977.55 

Running for 2020 with tariff 5
974.08 

Running for 2021 with tariff 5
922.32 

Running for 2022 with tariff 5
895.16 

Running for 2023 with tariff 5
950.11 

Running for 2024 with tariff 5
846.03 

Running for 2025 with tariff 5
921.13 

Running for 2017 with tariff 10
1041.83 

Running for 2018 with tariff 10
986.23 

Running for 2019 with tariff 10
977.55 

Running for 2020 with tariff 10
974.08 

Running for 2021 with tariff 10
922.32 

Running for 2022 with tariff 10
895.16 

Running for 2023 with tariff 10
9

## Now, convert the above optimization steps for a fixed network.
### Hint: create a function so that you can use in next steps.

Now convert above optimization steps for a fixed network into a function such that you pass on tariff, a fixed network design and exchange rate as parameters. The result should generate the data file consist of cost given the tariff and exchange rate.

Now
1) observe the past one year daily exchange rates and calculate mean fx and standard deviation,
2) using a chosen fixed network startegy, plot the exchange rates and cost distribution. This will give you an estimate of cost under volatile exchange rates.
Submit your github link/python notebook.